# OPEN DOTS · Phase 1 on a GPU
Qwen2.5-0.5B + 262k-slot HDC sparse memory + Dots ledger: learn 1,000 facts as 10 Dots, measure forgetting, exact removal (bit-for-bit) and reset, then LoRA and full fine-tuning baselines.

**v2:** context-robust DOTS + fairly tuned LoRA / full fine-tuning baselines. **Before Run all:** Runtime → Change runtime type → **T4 GPU**. Keep this tab open. About 1.5 h in total. The last cell downloads `phase1_v2_results.zip`.

In [ ]:
import os, subprocess
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'   # deterministic cuBLAS for exact removal
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total', '--format=csv'], capture_output=True, text=True).stdout)
!pip -q install "transformers==5.16.1" "peft==0.21.0" "datasets>=3.0" pytest
!pip -q uninstall -y torchao   # Colab's old torchao breaks peft LoRA
import torch, transformers; print('torch', torch.__version__, '| transformers', transformers.__version__, '| cuda', torch.cuda.is_available())
assert torch.cuda.is_available(), 'Switch the runtime to a GPU (Runtime > Change runtime type > T4 GPU)'
os.makedirs('/content/phase1/dots', exist_ok=True); os.makedirs('/content/phase1/tests', exist_ok=True); os.makedirs('/content/phase1/data', exist_ok=True)
%cd /content/phase1

In [ ]:
%%writefile dots/__init__.py
"""OPEN DOTS Phase 1: HDC sparse memory retrofit + Dots ledger."""
import os

os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")   # required for deterministic cuBLAS (exact removal)


In [ ]:
%%writefile dots/memory.py
"""HDC-addressed sparse memory for retrofitting a pretrained transformer.

Training-free addressing: the input is whitened with background statistics, projected by a
FIXED Gaussian matrix (SimHash-style, angle preserving), split in two halves and matched
against FIXED random bipolar sub-keys (product keys). Nothing in the address path is ever
trained, so slot addresses never drift and every write stays findable (and removable).

Values start at zero, so attaching the memory leaves the model's function unchanged.
Values are buffers: the only trainable tensor is a per-write-session copy of the rows
being written (see begin_write / end_write).
"""
import math

import torch
import torch.nn as nn
import torch.nn.functional as F


class HDCMemory(nn.Module):
    def __init__(self, d_model: int, n_sub: int = 512, key_dim: int = 128, topk: int = 32,
                 temp: float = 1.0, seed: int = 1234):
        super().__init__()
        self.d_model, self.n_sub, self.key_dim, self.topk, self.temp = d_model, n_sub, key_dim, topk, temp
        self.n_slots = n_sub * n_sub
        g = torch.Generator().manual_seed(seed)
        self.register_buffer("proj", torch.randn(d_model, 2 * key_dim, generator=g) / math.sqrt(d_model))
        self.register_buffer("sub1", torch.randint(0, 2, (n_sub, key_dim), generator=g).float() * 2 - 1)
        self.register_buffer("sub2", torch.randint(0, 2, (n_sub, key_dim), generator=g).float() * 2 - 1)
        self.register_buffer("mu", torch.zeros(d_model))
        self.register_buffer("sigma", torch.ones(d_model))
        self.register_buffer("values", torch.zeros(self.n_slots, d_model))
        self.register_buffer("bg_df", torch.zeros(self.n_slots))
        self.register_buffer("bg_mass", torch.zeros(self.n_slots))
        self.register_buffer("bg_docs_t", torch.zeros((), dtype=torch.long))
        self._write = None          # (slot -> row position map, trainable rows)
        self._record = None         # list collecting slot indices when recording reads

    @property
    def bg_docs(self) -> int:
        return int(self.bg_docs_t)

    def address(self, x: torch.Tensor):
        z = (x.float() - self.mu) / self.sigma
        q1, q2 = (z @ self.proj).split(self.key_dim, dim=-1)
        scale = 1.0 / math.sqrt(self.key_dim)
        s1, i1 = (q1 @ self.sub1.T * scale).topk(self.topk, dim=-1)
        s2, i2 = (q2 @ self.sub2.T * scale).topk(self.topk, dim=-1)
        cand = (s1.unsqueeze(-1) + s2.unsqueeze(-2)).flatten(-2)
        cand_idx = (i1.unsqueeze(-1) * self.n_sub + i2.unsqueeze(-2)).flatten(-2)
        best, pos = cand.topk(self.topk, dim=-1)
        return cand_idx.gather(-1, pos), F.softmax(best / self.temp, dim=-1)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        idx, w = self.address(x)
        if self._record is not None:
            self._record.append((idx.detach(), w.detach()))
        v = F.embedding(idx, self.values)
        if self._write is not None:
            row, rows = self._write
            r = row[idx]
            hit = r >= 0
            if bool(hit.any()):
                v = torch.where(hit.unsqueeze(-1), rows[r.clamp(min=0)], v)
        return (w.unsqueeze(-1) * v).sum(-2).to(x.dtype)

    # ---- write session ------------------------------------------------------------------
    def begin_write(self, slots: torch.Tensor) -> nn.Parameter:
        dev = self.values.device
        row = torch.full((self.n_slots,), -1, dtype=torch.long, device=dev)
        row[slots] = torch.arange(len(slots), device=dev)
        rows = nn.Parameter(self.values[slots].clone())
        self._write = (row, rows)
        return rows

    def end_write(self) -> torch.Tensor:
        _, rows = self._write
        self._write = None
        return rows.detach().clone()


In [ ]:
%%writefile dots/retrofit.py
"""Attach an HDCMemory to one decoder layer of a HF Qwen2-style model, calibrate it, and run
partial forwards (layers below the memory never change, so their output can be cached)."""
import torch
import torch.nn as nn

from .memory import HDCMemory


class MLPWithMemory(nn.Module):
    """Parallel branch: mlp(x) + memory(x). Memory values start at zero -> identical function."""

    def __init__(self, mlp: nn.Module, memory: HDCMemory):
        super().__init__()
        self.mlp, self.memory = mlp, memory

    def forward(self, x):
        return self.mlp(x) + self.memory(x)


def attach_memory(model, layer: int, memory: HDCMemory) -> HDCMemory:
    block = model.model.layers[layer]
    memory = memory.to(next(block.parameters()).device)     # follow the model onto GPU
    block.mlp = MLPWithMemory(block.mlp, memory)
    model.__dict__["dots_memory"] = memory        # plain attribute: not registered twice
    model.__dict__["dots_layer"] = layer
    return memory


def _layer_out(out):
    return out[0] if isinstance(out, tuple) else out


def _rope(model, h):
    pid = torch.arange(h.shape[1], device=h.device).unsqueeze(0)
    return model.model.rotary_emb(h, pid), pid


def causal_mask(h):
    """Explicit additive causal mask: correct for eager and SDPA attention alike."""
    T = h.shape[1]
    return torch.full((T, T), torch.finfo(h.dtype).min, device=h.device).triu(1)[None, None]


def run_layers(model, h, start: int, stop: int):
    pe, pid = _rope(model, h)
    mask = causal_mask(h)
    for blk in model.model.layers[start:stop]:
        h = _layer_out(blk(h, attention_mask=mask, position_embeddings=pe, position_ids=pid))
    return h


@torch.no_grad()
def layer_input(model, ids: torch.Tensor, layer: int) -> torch.Tensor:
    """Hidden state entering `layer` (causal attention, right padding assumed)."""
    ids = ids.to(model.model.embed_tokens.weight.device)
    return run_layers(model, model.model.embed_tokens(ids), 0, layer)


def forward_from_layer(model, h: torch.Tensor, layer: int, positions: torch.Tensor | None = None):
    """Logits from `layer` upward. `positions` (bool [B,T]) restricts the LM head to those tokens."""
    h = model.model.norm(run_layers(model, h, layer, len(model.model.layers)))
    return model.lm_head(h[positions] if positions is not None else h)


def _split(batch):
    """A batch is ids [B,T] or (ids, valid [B,T] bool) for right-padded data."""
    ids, valid = batch if isinstance(batch, tuple) else (batch, torch.ones_like(batch, dtype=torch.bool))
    return ids, valid.to(ids.device)


@torch.no_grad()
def calibrate(model, batches, chunk_docs: bool = True, stats_batches: int | None = None):
    """Set whitening stats (pass 1) and background slot document-frequency (pass 2).
    Padding positions (valid == False) contribute to neither."""
    mem, layer = model.dots_memory, model.dots_layer
    block = model.model.layers[layer]
    xs, cur = [], {}
    hook = block.mlp.register_forward_pre_hook(lambda m, a: xs.append(a[0][cur["valid"]].float()))
    try:
        for batch in batches[:stats_batches]:
            ids, cur["valid"] = _split(batch)
            cur["valid"] = cur["valid"].to(mem.values.device)
            run_layers(model, layer_input(model, ids, layer), layer, layer + 1)
    finally:
        hook.remove()
    x = torch.cat(xs)
    mem.mu.copy_(x.mean(0))
    mem.sigma.copy_(x.std(0).clamp_min(1e-5))
    mem.bg_df.zero_()
    mem.bg_mass.zero_()
    mem.bg_docs_t.zero_()
    for batch in batches:
        ids, valid = _split(batch)
        valid = valid.to(mem.values.device)
        mem._record = []
        try:
            run_layers(model, layer_input(model, ids, layer), layer, layer + 1)
            idx, w = mem._record[0]
        finally:
            mem._record = None
        for seq, ok in zip(idx, valid):
            mem.bg_df[seq[ok].unique()] += 1
        mem.bg_mass.index_add_(0, idx[valid].flatten(), w[valid].flatten())
        mem.bg_docs_t += idx.shape[0]


In [ ]:
%%writefile dots/qa.py
"""QA formatting, answer-only loss masks, and SQuAD-style scoring."""
import re
import string
from collections import Counter
from dataclasses import dataclass

import torch
from dataclasses import replace

PROMPT = "Question: {q}\nAnswer:"
TARGET = " {a}\n"
FEWSHOT = ("Question: What is the capital of Japan?\nAnswer: Tokyo\n"
           "Question: Which planet is known as the Red Planet?\nAnswer: Mars\n"
           "Question: Who painted the Mona Lisa?\nAnswer: Leonardo da Vinci\n")


# Training-only contexts (different demonstrations from the evaluation FEWSHOT prefix)
TRAIN_PREFIXES = (
    "Question: What is the chemical symbol for gold?\nAnswer: Au\n"
    "Question: Which ocean is the largest?\nAnswer: Pacific Ocean\n",
    "Question: Who invented the telephone?\nAnswer: Alexander Graham Bell\n"
    "Question: What is the tallest mountain on Earth?\nAnswer: Mount Everest\n"
    "Question: How many legs does a spider have?\nAnswer: 8\n",
)


@dataclass(frozen=True)
class QAExample:
    question: str
    answers: list
    prefix: str = ""

    def prompt(self) -> str:
        return self.prefix + PROMPT.format(q=self.question)

    def target(self) -> str:
        return TARGET.format(a=self.answers[0])


def augment(examples, prefixes=TRAIN_PREFIXES):
    """Each fact in its plain form plus one copy per demonstration context."""
    out = []
    for ex in examples:
        out += [ex] + [replace(ex, prefix=p) for p in prefixes]
    return out


def build_prompt(ex, prefix: str = "") -> str:
    """Evaluation prompt; training never uses a prefix."""
    return prefix + ex.prompt()


def encode_qa(tok, examples, max_len: int = 96):
    """Right-padded ids and a mask marking positions whose NEXT token is an answer token."""
    rows, spans = [], []
    for ex in examples:
        p = tok.encode(ex.prompt(), add_special_tokens=False)
        t = tok.encode(ex.target(), add_special_tokens=False)
        ids = (p + t)[:max_len]
        rows.append(ids)
        spans.append((len(p) - 1, min(len(ids), len(p) + len(t)) - 1))
    T = max(len(r) for r in rows)
    pad = tok.pad_token_id if tok.pad_token_id is not None else 0
    x = torch.full((len(rows), T), pad, dtype=torch.long)
    mask = torch.zeros(len(rows), T)
    for i, (r, (a, b)) in enumerate(zip(rows, spans)):
        x[i, :len(r)] = torch.tensor(r)
        mask[i, a:b] = 1.0
    return x, mask


def valid_positions(mask: torch.Tensor) -> torch.Tensor:
    """All positions up to and including the last answer token (excludes right padding)."""
    ar = torch.arange(mask.shape[1], device=mask.device)
    last = (mask * ar).max(1).values + 1
    return ar.unsqueeze(0) <= last.unsqueeze(1)


def normalize_answer(s: str) -> str:
    s = s.lower()
    s = "".join(" " if ch in string.punctuation else ch for ch in s)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())


def exact_match(pred: str, golds) -> float:
    p = normalize_answer(pred)
    return float(any(p == normalize_answer(g) for g in golds))


def token_f1(pred: str, golds) -> float:
    def f1(a, b):
        pa, pb = normalize_answer(a).split(), normalize_answer(b).split()
        common = sum((Counter(pa) & Counter(pb)).values())
        if not pa or not pb or common == 0:
            return 0.0
        prec, rec = common / len(pa), common / len(pb)
        return 2 * prec * rec / (prec + rec)
    return max(f1(pred, g) for g in golds)


In [ ]:
%%writefile dots/ledger.py
"""Dots ledger: every memory write is a Dot (source, time, seed, read set, write set, delta).

Canonical memory = base + sum of Dot deltas in Dot order, recomputed (never subtracted).
Only rows ever written are recomputed, so it stays cheap for a 262k-slot table.
  reset         -> no Dots (bitwise base)
  fast removal  -> drop one Dot's delta, no retraining
  exact removal -> drop it and replay only Dots whose reads touched rows it (transitively)
                   influenced; result is bitwise identical to never having learned it
"""
import hashlib
import time
from dataclasses import dataclass, field

import torch


@dataclass
class Dot:
    dot_id: int
    source: str
    data: list
    seed: int
    write_slots: torch.Tensor
    read_mask: torch.Tensor
    delta: torch.Tensor
    created: float = field(default_factory=time.time)

    def nbytes(self) -> int:
        return self.delta.numel() * self.delta.element_size() + self.write_slots.numel() * 8


class Ledger:
    def __init__(self, memory):
        self.memory = memory
        v = memory.values
        self.base = None if int(torch.count_nonzero(v)) == 0 else v.clone()
        self.base_hash = self.state_hash()
        self.dots: list[Dot] = []
        self._dirty = torch.zeros(memory.n_slots, dtype=torch.bool, device=v.device)
        self._next = 0

    def _base_rows(self, rows):
        if self.base is None:
            return torch.zeros(len(rows), self.memory.d_model, device=self.memory.values.device)
        return self.base[rows].clone()

    def materialize(self, dots=None) -> None:
        dots = self.dots if dots is None else dots
        touched = self._dirty.clone()
        for d in dots:
            touched[d.write_slots] = True
        rows = touched.nonzero().flatten()
        v = self.memory.values
        with torch.no_grad():
            v[rows] = self._base_rows(rows)
            for d in dots:
                v[d.write_slots] += d.delta
        self._dirty = touched

    def state_hash(self) -> str:
        return hashlib.sha256(self.memory.values.detach().cpu().numpy().tobytes()).hexdigest()

    def next_id(self) -> int:
        self._next += 1
        return self._next

    def record(self, dot: Dot) -> None:
        self.dots.append(dot)
        self.materialize()

    def reset(self) -> None:
        self.dots = []
        self.materialize()

    def owned(self, dots=None) -> torch.Tensor:
        m = torch.zeros(self.memory.n_slots, dtype=torch.bool, device=self.memory.values.device)
        for d in (self.dots if dots is None else dots):
            m[d.write_slots] = True
        return m

    def find(self, source: str):
        return [d for d in self.dots if d.source == source]

    def affected_by(self, dot_id: int) -> list[int]:
        tainted, out = None, []
        for d in self.dots:
            if d.dot_id == dot_id:
                tainted = torch.zeros(self.memory.n_slots, dtype=torch.bool, device=d.read_mask.device)
                tainted[d.write_slots] = True
            elif tainted is not None and bool((d.read_mask & tainted).any()):
                out.append(d.dot_id)
                tainted[d.write_slots] = True
        return out

    def fanout(self) -> dict:
        return {d.dot_id: len(self.affected_by(d.dot_id)) for d in self.dots}

    def remove(self, dot_id: int, writer=None, exact: bool = True) -> dict:
        t0 = time.perf_counter()
        affected = set(self.affected_by(dot_id)) if exact else set()
        if affected and writer is None:
            raise ValueError("exact removal with dependent Dots needs a writer to replay them")
        kept: list[Dot] = []
        for d in self.dots:
            if d.dot_id == dot_id:
                continue
            if d.dot_id in affected:
                self.materialize(kept)
                d = writer.retrain(d, kept)
            kept.append(d)
        self.dots = kept
        self.materialize()
        return {"removed": dot_id, "mode": "exact" if exact else "fast", "replayed": sorted(affected),
                "seconds": round(time.perf_counter() - t0, 2)}


In [ ]:
%%writefile dots/writer.py
"""DotWriter: learn a batch of QA facts as one Dot by writing only selected memory rows.

1. Encode facts; cache the hidden state entering the memory layer (lower layers never change).
2. Record which slots the facts read (read set) and how often (TF).
3. Choose write slots by TF-IDF against background usage, excluding slots owned by earlier Dots.
4. Plain SGD on those rows only, answer-token loss, deterministic seed from the data.
Backward runs only through the layers above the memory; nothing else in the model changes.
"""
import hashlib
from dataclasses import dataclass, replace

import torch
import torch.nn.functional as F

from .ledger import Dot, Ledger
from .qa import encode_qa, valid_positions
from .retrofit import forward_from_layer, layer_input, run_layers


@dataclass(frozen=True)
class WriterConfig:
    write_slots: int = 2000
    steps: int = 60
    lr: float = 1.0
    batch_size: int = 16
    use_ownership: bool = True
    max_len: int = 160
    optimizer: str = "sgd"          # "sgd" or "adam"; a fresh optimizer per Dot keeps removal exact
    protect_quantile: float | None = None   # never write slots whose background read mass is above this quantile
    anchor_weight: float = 0.0      # penalty on row changes, weighted by how much background reads each row


def deterministic_ce(logits, targets):
    """Cross-entropy via log_softmax * one-hot: no atomics, bit-reproducible on GPU too."""
    logp = F.log_softmax(logits, dim=-1)
    return -(logp * F.one_hot(targets, logits.shape[-1]).to(logp.dtype)).sum(-1).mean()


class DotWriter:
    def __init__(self, model, tok, layer: int, cfg: WriterConfig = WriterConfig()):
        self.model, self.tok, self.layer, self.cfg = model, tok, layer, cfg
        self.mem = model.dots_memory

    @staticmethod
    def seed_for(examples) -> int:
        blob = "\x1e".join(e.prefix + e.question + "\x1f" + "\x1f".join(e.answers) for e in examples)
        return int(hashlib.sha256(blob.encode()).hexdigest()[:8], 16)

    @torch.no_grad()
    def _reads(self, h, valid):
        self.mem._record = []
        try:
            run_layers(self.model, h, self.layer, self.layer + 1)
            idx = self.mem._record[0][0]
        finally:
            self.mem._record = None
        sel = idx[valid]
        counts = torch.bincount(sel.flatten(), minlength=self.mem.n_slots).float()
        return counts > 0, counts

    def _select(self, counts, owned):
        eligible = counts > 0
        if self.cfg.use_ownership:
            eligible &= ~owned
        if self.cfg.protect_quantile is not None:
            eligible &= self.mem.bg_mass <= torch.quantile(self.mem.bg_mass, self.cfg.protect_quantile)
        idf = torch.log((self.mem.bg_docs + 1.0) / (self.mem.bg_df + 1.0))
        score = torch.where(eligible, counts * idf, torch.full_like(counts, -1.0))
        k = min(self.cfg.write_slots, int(eligible.sum()))
        return score.topk(k).indices.sort().values

    def _prepare(self, examples):
        ids, mask = encode_qa(self.tok, examples, self.cfg.max_len)
        dev = self.mem.values.device
        ids, mask = ids.to(dev), mask.to(dev)
        h = layer_input(self.model, ids, self.layer)
        return ids, mask, h

    def _train(self, ids, mask, h, slots, seed):
        was_det = torch.are_deterministic_algorithms_enabled()
        torch.use_deterministic_algorithms(True, warn_only=True)   # exact removal needs bit-reproducible replay
        rows = self.mem.begin_write(slots)
        flags = {n: p.requires_grad for n, p in self.model.named_parameters()}
        for p in self.model.parameters():
            p.requires_grad_(False)
        opt = (torch.optim.Adam([rows], lr=self.cfg.lr) if self.cfg.optimizer == "adam"
               else torch.optim.SGD([rows], lr=self.cfg.lr))
        gen = torch.Generator().manual_seed(seed)
        targets_all = ids.roll(-1, dims=1)
        start = rows.detach().clone()
        used = self.mem.bg_mass[self.mem.bg_mass > 0]
        importance = (self.mem.bg_mass[slots] / (used.mean() if len(used) else 1.0)).unsqueeze(1)
        try:
            for _ in range(self.cfg.steps):
                bi = torch.randint(0, ids.shape[0], (min(self.cfg.batch_size, ids.shape[0]),), generator=gen).to(ids.device)
                pos = mask[bi].bool()
                logits = forward_from_layer(self.model, h[bi], self.layer, positions=pos)
                loss = deterministic_ce(logits, targets_all[bi][pos])
                if self.cfg.anchor_weight > 0:
                    loss = loss + self.cfg.anchor_weight * (importance * (rows - start) ** 2).sum(1).mean()
                opt.zero_grad(set_to_none=True)
                loss.backward()
                opt.step()
        finally:
            after = self.mem.end_write()
            for n, p in self.model.named_parameters():
                p.requires_grad_(flags[n])
            torch.use_deterministic_algorithms(was_det)
        return after

    def learn(self, examples, source: str, ledger: Ledger) -> Dot:
        ledger.materialize()
        ids, mask, h = self._prepare(examples)
        read_mask, counts = self._reads(h, valid_positions(mask))
        slots = self._select(counts, ledger.owned())
        seed = self.seed_for(examples)
        before = self.mem.values[slots].clone()
        after = self._train(ids, mask, h, slots, seed)
        dot = Dot(ledger.next_id(), source, list(examples), seed, slots, read_mask, after - before)
        ledger.record(dot)
        return dot

    def retrain(self, dot: Dot, prior) -> Dot:
        """Replay `dot` as if `prior` were the whole history (state already materialized)."""
        ids, mask, h = self._prepare(dot.data)
        _, counts = self._reads(h, valid_positions(mask))
        owned = torch.zeros(self.mem.n_slots, dtype=torch.bool, device=self.mem.values.device)
        for d in prior:
            owned[d.write_slots] = True
        slots = self._select(counts, owned)
        before = self.mem.values[slots].clone()
        after = self._train(ids, mask, h, slots, dot.seed)
        return replace(dot, write_slots=slots, delta=after - before)


In [ ]:
%%writefile dots/evaluate.py
"""Greedy QA generation (EM / F1) and perplexity."""
import torch
import torch.nn.functional as F

from .qa import build_prompt, exact_match, token_f1


def clean_generation(text: str) -> str:
    for line in text.split("\n"):
        if line.strip():
            return line.strip()
    return ""


@torch.no_grad()
def generate_answers(model, tok, examples, batch_size: int = 32, max_new_tokens: int = 12, prefix: str = ""):
    tok.padding_side = "left"
    out = []
    for i in range(0, len(examples), batch_size):
        chunk = examples[i:i + batch_size]
        enc = tok([build_prompt(e, prefix) for e in chunk], return_tensors="pt", padding=True,
                  add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        texts = tok.batch_decode(gen[:, enc.input_ids.shape[1]:], skip_special_tokens=True)
        out += [clean_generation(t) for t in texts]
    return out


def score(preds, examples) -> dict:
    em = [exact_match(p, e.answers) for p, e in zip(preds, examples)]
    f1 = [token_f1(p, e.answers) for p, e in zip(preds, examples)]
    n = max(1, len(examples))
    return {"em": sum(em) / n, "f1": sum(f1) / n, "n": len(examples), "per_item_em": em}


def evaluate_qa(model, tok, examples, **kw) -> dict:
    return score(generate_answers(model, tok, examples, **kw), examples)


@torch.no_grad()
def perplexity(model, batches) -> float:
    total, count = 0.0, 0
    for x in batches:
        x = x.to(model.device)
        logits = model(x).logits[:, :-1]
        total += float(F.cross_entropy(logits.reshape(-1, logits.shape[-1]), x[:, 1:].reshape(-1), reduction="sum"))
        count += x[:, 1:].numel()
    return float(torch.exp(torch.tensor(total / count)))


In [ ]:
%%writefile dots/common.py
"""Shared loading helpers for Phase 1 scripts."""
import json
from pathlib import Path

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

from .memory import HDCMemory
from .qa import FEWSHOT, QAExample
from .retrofit import attach_memory, calibrate

ROOT = Path(__file__).resolve().parent.parent
DATA, RUNS = ROOT / "data", ROOT / "runs"
MODEL = "Qwen/Qwen2.5-0.5B"


DEVICE = "cuda" if torch.cuda.is_available() else "cpu"


def load_base():
    """fp32; on GPU use eager attention so training is bit-reproducible (exact removal)."""
    tok = AutoTokenizer.from_pretrained(MODEL)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token
    kw = {"attn_implementation": "eager"} if DEVICE == "cuda" else {}
    try:
        model = AutoModelForCausalLM.from_pretrained(MODEL, dtype=torch.float32, **kw)
    except TypeError:
        model = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32, **kw)
    return model.to(DEVICE).eval(), tok


def load_data():
    d = json.loads((DATA / "phase1_data.json").read_text(encoding="utf-8"))
    ex = lambda xs: [QAExample(r["question"], r["answers"]) for r in xs]
    chunks = torch.load(DATA / "wikitext_chunks.pt")
    return {"teach": ex(d["teach"]), "known": ex(d["known"]), "nq": ex(d["nq"]), "base": d["base"],
            "background": chunks["background"], "heldout": chunks["heldout"]}


def qa_background(tok, n_zero=1000, n_few=500, seed=7, batch=32):
    """Chat-format background: NQ-open TRAIN Q&A (disjoint from every evaluation set), with and
    without the few-shot prefix, so slots used by the Q&A format itself count as common."""
    import random
    from datasets import load_dataset
    ds = load_dataset("google-research-datasets/nq_open", split="train")
    idx = random.Random(seed).sample(range(len(ds)), n_zero + n_few)
    texts = []
    for j, i in enumerate(idx):
        ex = QAExample(ds[i]["question"].strip() + "?", list(ds[i]["answer"]))
        texts.append((FEWSHOT if j >= n_zero else "") + ex.prompt() + ex.target())
    tok.padding_side = "right"
    out = []
    for k in range(0, len(texts), batch):
        enc = tok(texts[k:k + batch], return_tensors="pt", padding=True, add_special_tokens=False)
        out.append((enc.input_ids, enc.attention_mask.bool()))
    return out


def attach_calibrated(model, data, layer=15, n_sub=512, key_dim=128, topk=32, temp=1.0, tok=None):
    """tok given -> background = WikiText + chat-format Q&A; otherwise WikiText only."""
    mem = attach_memory(model, layer, HDCMemory(model.config.hidden_size, n_sub, key_dim, topk, temp))
    RUNS.mkdir(exist_ok=True)
    tag = "_qa2" if tok is not None else "_v2"
    path = RUNS / f"calib_L{layer}_s{n_sub}_k{key_dim}_t{topk}{tag}.pt"
    if path.exists():
        st = torch.load(path)
        for k in ("mu", "sigma", "bg_df", "bg_mass", "bg_docs_t"):
            getattr(mem, k).copy_(st[k])
    else:
        batches = list(data["background"].split(16))
        if tok is not None:
            qa_b = qa_background(tok)
            mixed = [b for pair in zip(batches, qa_b) for b in pair]
            batches = mixed + batches[len(qa_b):] + qa_b[len(batches):]
        calibrate(model, batches, stats_batches=12)
        torch.save({k: getattr(mem, k).clone() for k in ("mu", "sigma", "bg_df", "bg_mass", "bg_docs_t")}, path)
    return mem


def dots_of(teach, n_dots):
    size = len(teach) // n_dots
    return [teach[i * size:(i + 1) * size] for i in range(n_dots)]


In [ ]:
%%writefile run_dots.py
"""Phase 1 main run: Qwen2.5-0.5B + HDC sparse memory + Dots ledger.

Learns 1,000 TriviaQA facts as 10 Dots, then measures:
  acquisition (EM on taught facts), retention (known facts, NQ-open F1, WikiText-2 perplexity),
  taint fan-out, fast removal, exact removal (bitwise vs a never-learned counterfactual), reset.
"""
import argparse
import copy
import json
import sys
import time
from pathlib import Path

import torch

sys.path.insert(0, str(Path(__file__).parent))
from dots.common import RUNS, attach_calibrated, dots_of, load_base, load_data  # noqa: E402
from dots.evaluate import evaluate_qa, perplexity  # noqa: E402
from dots.ledger import Ledger  # noqa: E402
from dots.qa import FEWSHOT, augment  # noqa: E402
from dots.writer import DotWriter, WriterConfig  # noqa: E402

torch.set_num_threads(6)


def log(*a):
    print(time.strftime("%H:%M:%S"), *a, flush=True)


def qa(model, tok, xs, prefix=""):
    r = evaluate_qa(model, tok, xs, prefix=prefix)
    return {"em": round(r["em"], 4), "f1": round(r["f1"], 4), "n": r["n"]}


def full_eval(model, tok, data, groups):
    out = {"dots": [qa(model, tok, g)["em"] for g in groups]}
    out["teach_em"] = round(sum(out["dots"]) / len(out["dots"]), 4)
    out["teach_fewshot_dots1_2"] = qa(model, tok, groups[0] + groups[1], prefix=FEWSHOT)
    out["known"] = qa(model, tok, data["known"], prefix=FEWSHOT)
    out["nq"] = qa(model, tok, data["nq"], prefix=FEWSHOT)
    out["ppl"] = round(perplexity(model, list(data["heldout"].split(10))), 4)
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--lr", type=float, required=True)
    ap.add_argument("--steps", type=int, default=60)
    ap.add_argument("--slots", type=int, default=3000)
    ap.add_argument("--layer", type=int, default=15)
    ap.add_argument("--dots", type=int, default=10)
    ap.add_argument("--tag", default="main")
    ap.add_argument("--opt", default="adam")
    ap.add_argument("--protect", type=float, default=None)
    ap.add_argument("--anchor", type=float, default=0.0)
    ap.add_argument("--augment", action="store_true", help="train each fact in extra demonstration contexts")
    ap.add_argument("--smoke", action="store_true", help="tiny end-to-end check")
    a = ap.parse_args()
    res = {"args": vars(a)}
    model, tok = load_base()
    data = load_data()
    if a.smoke:
        data = {**data, "teach": data["teach"][:12], "known": data["known"][:6], "nq": data["nq"][:6],
                "heldout": data["heldout"][:2]}
        a.dots, a.steps = 3, 2
    probe = tok(["Question: Who wrote Hamlet?\nAnswer:", "The capital of France is"], return_tensors="pt", padding=True).to(model.device)
    with torch.no_grad():
        ref = model(**probe).logits
    attach_calibrated(model, data, layer=a.layer, tok=tok)
    with torch.no_grad():
        res["function_preserving"] = bool(torch.equal(ref, model(**probe).logits))
    res["base"] = {**data["base"], "ppl": round(perplexity(model, list(data["heldout"].split(10))), 4)}
    log("attached; function preserving:", res["function_preserving"], "| base ppl", res["base"]["ppl"])

    groups = dots_of(data["teach"], a.dots)
    train_groups = [augment(g) for g in groups] if a.augment else groups
    ledger = Ledger(model.dots_memory)
    writer = DotWriter(model, tok, a.layer, WriterConfig(write_slots=a.slots, steps=a.steps, lr=a.lr, optimizer=a.opt, protect_quantile=a.protect, anchor_weight=a.anchor))
    ckpt = RUNS / f"ckpt_{a.tag}.pt"
    if ckpt.exists() and not a.smoke:      # resume after an interruption: skip learning + first eval
        st = torch.load(ckpt, weights_only=False)
        ledger.dots, ledger._next = st["dots"], st["next"]
        ledger.materialize()
        res.update(st["res"])
        log("resumed from checkpoint:", len(ledger.dots), "dots")
    else:
        times = []
        for i, g in enumerate(groups):
            t = time.time()
            d = writer.learn(train_groups[i], f"triviaqa-dot-{i + 1}", ledger)
            times.append(round(time.time() - t, 1))
            log(f"dot {i + 1}: {len(d.write_slots)} slots, {d.nbytes() / 2**20:.1f} MiB, {times[-1]}s")
            if i == 0:
                res["dot1_right_after"] = qa(model, tok, g)
                log("dot1 recall right after learning:", res["dot1_right_after"])
        res["dot_seconds"] = times
        res["after_all"] = full_eval(model, tok, data, groups)
        log("after all dots:", {k: v for k, v in res["after_all"].items() if k != "dots"})
        fan = ledger.fanout()
        res["fanout"] = {"per_dot": fan, "mean_fraction": round(sum(v / max(1, a.dots - i - 1) for i, v in enumerate(fan.values()) if i < a.dots - 1) / max(1, a.dots - 1), 3)}
        res["ledger_mib"] = round(sum(d.nbytes() for d in ledger.dots) / 2**20, 1)
        log("fan-out:", res["fanout"])
        if not a.smoke:
            torch.save({"dots": ledger.dots, "next": ledger._next, "res": res}, ckpt)

    saved = copy.deepcopy(ledger.dots)
    d1 = ledger.dots[0].dot_id
    probe_sets = {"dot1": (groups[0], ""), "dot2": (groups[1], ""), "known": (data["known"], FEWSHOT)}
    rep = ledger.remove(d1, exact=False)
    res["fast_remove_dot1"] = {**rep, **{k: qa(model, tok, v, p)["em"] for k, (v, p) in probe_sets.items()}}
    log("fast removal:", res["fast_remove_dot1"])
    ledger.dots = copy.deepcopy(saved)
    ledger.materialize()
    rep = ledger.remove(d1, writer=writer, exact=True)
    exact_state = model.dots_memory.values.clone()
    res["exact_remove_dot1"] = {**rep, **{k: qa(model, tok, v, p)["em"] for k, (v, p) in probe_sets.items()}}
    log("exact removal:", {k: v for k, v in res["exact_remove_dot1"].items() if k != "replayed"})
    ledger.reset()
    t = time.time()
    for i, g in enumerate(train_groups[1:], start=2):
        writer.learn(g, f"triviaqa-dot-{i}", ledger)
    res["counterfactual_seconds"] = round(time.time() - t, 1)
    res["exact_equals_never_learned"] = bool(torch.equal(exact_state, model.dots_memory.values))
    del exact_state
    ledger.reset()
    with torch.no_grad():
        res["reset_hash_equal"] = ledger.state_hash() == ledger.base_hash
        res["reset_logits_equal_original"] = bool(torch.equal(ref, model(**probe).logits))
    log("exact == never learned:", res["exact_equals_never_learned"], "| reset:", res["reset_hash_equal"], res["reset_logits_equal_original"])
    (RUNS / f"dots_{a.tag}{'_smoke' if a.smoke else ''}.json").write_text(json.dumps(res, indent=1))
    log("saved")


if __name__ == "__main__":
    main()


In [ ]:
%%writefile run_baselines.py
"""Yardstick baselines on the same 10 fact chunks: full fine-tuning (Adafactor) and LoRA.

Same answer-only loss, batch size and steps per chunk as the Dots run. Embeddings (tied with
the LM head) stay frozen in full FT to fit 8 GB of RAM; every transformer block is trained.
"""
import argparse
import json
import sys
import time
from pathlib import Path

import torch
import torch.nn.functional as F

sys.path.insert(0, str(Path(__file__).parent))
from dots.common import RUNS, dots_of, load_base, load_data  # noqa: E402
from dots.qa import augment, encode_qa  # noqa: E402
from run_dots import full_eval, log, qa  # noqa: E402

torch.set_num_threads(6)


def train_chunk(model, tok, examples, opt, steps, bs, seed):
    dev = next(model.parameters()).device
    ids, mask = encode_qa(tok, examples)
    ids, mask = ids.to(dev), mask.to(dev)
    targets = ids.roll(-1, dims=1)
    gen = torch.Generator().manual_seed(seed)
    base = model.get_base_model() if hasattr(model, "get_base_model") else model
    for _ in range(steps):
        bi = torch.randint(0, ids.shape[0], (min(bs, ids.shape[0]),), generator=gen).to(dev)
        pos = mask[bi].bool()
        h = base.model(input_ids=ids[bi]).last_hidden_state
        loss = F.cross_entropy(base.lm_head(h[pos]), targets[bi][pos])
        opt.zero_grad(set_to_none=True)
        loss.backward()
        opt.step()


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--method", choices=["fullft", "lora"], required=True)
    ap.add_argument("--lr", type=float, required=True)
    ap.add_argument("--steps", type=int, default=60)
    ap.add_argument("--dots", type=int, default=10)
    ap.add_argument("--augment", action="store_true")
    ap.add_argument("--smoke", action="store_true", help="tiny end-to-end check")
    a = ap.parse_args()
    model, tok = load_base()
    data = load_data()
    if a.smoke:
        data = {**data, "teach": data["teach"][:12], "known": data["known"][:6], "nq": data["nq"][:6],
                "heldout": data["heldout"][:2]}
        a.dots, a.steps = 3, 2
    if a.method == "lora":
        from peft import LoraConfig, get_peft_model
        model = get_peft_model(model, LoraConfig(r=32, lora_alpha=64, lora_dropout=0.0, target_modules=[
            "q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]))
        opt = torch.optim.AdamW([p for p in model.parameters() if p.requires_grad], lr=a.lr, weight_decay=0.0)
    else:
        from transformers.optimization import Adafactor
        model.model.embed_tokens.weight.requires_grad_(False)
        opt = Adafactor([p for p in model.parameters() if p.requires_grad], lr=a.lr,
                        scale_parameter=False, relative_step=False, warmup_init=False)
    model.train()
    groups = dots_of(data["teach"], a.dots)
    res = {"args": vars(a), "trainable_params": sum(p.numel() for p in model.parameters() if p.requires_grad)}
    times = []
    for i, g in enumerate(groups):
        g_train = augment(g) if a.augment else g
        t = time.time()
        train_chunk(model, tok, g_train, opt, a.steps, 16, seed=1000 + i)
        times.append(round(time.time() - t, 1))
        log(f"{a.method} chunk {i + 1}: {times[-1]}s")
        if i == 0:
            model.eval()
            res["dot1_right_after"] = qa(model, tok, g)
            log("dot1 recall right after:", res["dot1_right_after"])
            model.train()
    res["chunk_seconds"] = times
    model.eval()
    res["after_all"] = full_eval(model, tok, data, groups)
    log("after all:", {k: v for k, v in res["after_all"].items() if k != "dots"})
    (RUNS / f"baseline_{a.method}_lr{a.lr:g}{'_aug' if a.augment else ''}{'_smoke' if a.smoke else ''}.json").write_text(json.dumps(res, indent=1))


if __name__ == "__main__":
    main()


In [ ]:
%%writefile make_chunks.py
"""Rebuild WikiText-2 token chunks (background 400x128, held-out 100x128) exactly as prepare_data.py did."""
import torch
from datasets import load_dataset
from transformers import AutoTokenizer

from dots.common import DATA, MODEL


def chunks(tok, texts, size, count):
    ids = tok("\n".join(t for t in texts if t.strip()), add_special_tokens=False).input_ids
    n = min(count, len(ids) // size)
    return torch.tensor(ids[: n * size]).view(n, size)


if __name__ == "__main__":
    tok = AutoTokenizer.from_pretrained(MODEL)
    wt = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
    DATA.mkdir(exist_ok=True)
    out = {"background": chunks(tok, wt["train"]["text"], 128, 400), "heldout": chunks(tok, wt["test"]["text"], 128, 100)}
    torch.save(out, DATA / "wikitext_chunks.pt")
    print({k: tuple(v.shape) for k, v in out.items()})


In [ ]:
%%writefile conftest.py
import sys
from pathlib import Path

sys.path.insert(0, str(Path(__file__).parent))


In [ ]:
%%writefile tests/test_retrofit.py
"""Phase 1 retrofit tests on a tiny random Qwen2 (fast, CPU)."""
import pytest
import torch
from transformers import Qwen2Config, Qwen2ForCausalLM

from dots.memory import HDCMemory
from dots.retrofit import attach_memory, calibrate, layer_input, forward_from_layer
from dots.qa import QAExample, encode_qa, normalize_answer, exact_match, token_f1
from dots.writer import DotWriter, WriterConfig
from dots.ledger import Ledger

LAYER = 2


def tiny_model(seed=0):
    torch.manual_seed(seed)
    cfg = Qwen2Config(vocab_size=300, hidden_size=64, intermediate_size=128, num_hidden_layers=4,
                      num_attention_heads=4, num_key_value_heads=2, max_position_embeddings=128,
                      tie_word_embeddings=True)
    return Qwen2ForCausalLM(cfg).eval()


def toks(n, length, seed=0):
    g = torch.Generator().manual_seed(seed)
    return torch.randint(5, 300, (n, length), generator=g)


@pytest.fixture()
def model():
    m = tiny_model()
    attach_memory(m, LAYER, HDCMemory(64, n_sub=16, key_dim=16, topk=4))
    calibrate(m, [toks(8, 12, seed=1)], chunk_docs=True)
    return m


def facts(offset=0, n=4):
    return [QAExample(question=f"q{offset + i} what is {offset + i}?", answers=[f"a{offset + i}"]) for i in range(n)]


class FakeTok:
    """Deterministic char-level tokenizer stand-in (ids 5..254)."""
    eos_token_id = 1
    pad_token_id = 0

    def encode(self, s, add_special_tokens=False):
        return [5 + (ord(c) % 250) for c in s]


# ---- memory --------------------------------------------------------------------------
def test_memory_keys_fixed_bipolar_and_values_zero():
    mem = HDCMemory(32, n_sub=8, key_dim=8, topk=2)
    for name in ("sub1", "sub2"):
        k = getattr(mem, name)
        assert not k.requires_grad and set(k.unique().tolist()) <= {-1.0, 1.0}
    assert mem.n_slots == 64 and torch.count_nonzero(mem.values) == 0
    assert list(mem.parameters()) == []          # nothing trainable outside a write session


def test_memory_addressing_deterministic_topk():
    mem = HDCMemory(32, n_sub=8, key_dim=8, topk=3)
    x = torch.randn(2, 5, 32)
    i1, w1 = mem.address(x)
    i2, w2 = mem.address(x)
    assert torch.equal(i1, i2) and torch.equal(w1, w2) and i1.shape == (2, 5, 3)
    assert torch.allclose(w1.sum(-1), torch.ones(2, 5))


# ---- retrofit ------------------------------------------------------------------------
def test_attach_is_function_preserving():
    base = tiny_model()
    x = toks(3, 10, seed=2)
    with torch.no_grad():
        ref = base(x).logits
    attach_memory(base, LAYER, HDCMemory(64, n_sub=16, key_dim=16, topk=4))
    with torch.no_grad():
        out = base(x).logits
    assert torch.equal(ref, out)


def test_calibration_sets_whitening_and_background_counts(model):
    mem = model.dots_memory
    assert mem.mu.abs().sum() > 0 and torch.all(mem.sigma > 0)
    assert mem.bg_df.sum() > 0 and mem.bg_docs == 8


def test_forward_from_layer_matches_full_forward(model):
    x = toks(3, 10, seed=3)
    with torch.no_grad():
        full = model(x).logits
        h = layer_input(model, x, LAYER)
        part = forward_from_layer(model, h, LAYER)
    assert torch.allclose(full, part, atol=1e-5)


# ---- QA utilities ----------------------------------------------------------------------
def test_encode_qa_masks_only_answer_tokens():
    ids, mask = encode_qa(FakeTok(), [QAExample("q1?", ["ab"])])
    prompt_len = len(FakeTok().encode("Question: q1?\nAnswer:"))
    target = mask[0].nonzero().flatten().tolist()
    # loss positions predict the answer tokens " ab" and the newline
    assert target[0] == prompt_len - 1 and len(target) == len(FakeTok().encode(" ab\n"))


def test_answer_normalization_and_scores():
    assert normalize_answer("The  Eiffel-Tower.") == "eiffel tower"
    assert exact_match("the eiffel tower", ["Eiffel Tower"]) == 1.0
    assert token_f1("eiffel", ["Eiffel Tower"]) == pytest.approx(2 / 3)


# ---- writer + ledger -------------------------------------------------------------------
def test_dot_writes_only_selected_slots_and_nothing_else(model):
    ledger = Ledger(model.dots_memory)
    writer = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=20, steps=3, lr=0.5, batch_size=2))
    params_before = {n: p.detach().clone() for n, p in model.named_parameters()}
    before = model.dots_memory.values.clone()
    dot = writer.learn(facts(0), source="s0", ledger=ledger)
    changed = (model.dots_memory.values != before).any(-1).nonzero().flatten().tolist()
    assert changed and set(changed) <= set(dot.write_slots.tolist()) and len(dot.write_slots) <= 20
    for n, p in model.named_parameters():
        assert torch.equal(p, params_before[n]), n


def test_reset_is_bitwise(model):
    mem = model.dots_memory
    ledger = Ledger(mem)
    h0 = ledger.state_hash()
    DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=10, steps=2, lr=0.5, batch_size=2)).learn(facts(0), "s", ledger)
    assert ledger.state_hash() != h0
    ledger.reset()
    assert ledger.state_hash() == h0 and ledger.dots == []


def test_ownership_keeps_dots_disjoint(model):
    ledger = Ledger(model.dots_memory)
    w = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=2, lr=0.5, batch_size=2))
    a = w.learn(facts(0), "A", ledger)
    b = w.learn(facts(10), "B", ledger)
    assert not set(a.write_slots.tolist()) & set(b.write_slots.tolist())


def test_exact_removal_equals_never_learned(model):
    ledger = Ledger(model.dots_memory)
    w = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=3, lr=0.5, batch_size=2))
    a = w.learn(facts(0), "A", ledger)
    w.learn(facts(10), "B", ledger)
    w.learn(facts(20), "C", ledger)
    ledger.remove(a.dot_id, writer=w, exact=True)
    after = model.dots_memory.values.clone()
    ledger.reset()
    w.learn(facts(10), "B", ledger)
    w.learn(facts(20), "C", ledger)
    assert torch.equal(after, model.dots_memory.values)


def test_fanout_counts_transitive_dependents(model):
    ledger = Ledger(model.dots_memory)
    w = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=1, lr=0.5, batch_size=2))
    dots = [w.learn(facts(10 * i), f"D{i}", ledger) for i in range(3)]
    fan = ledger.fanout()
    assert set(fan) == {d.dot_id for d in dots} and fan[dots[-1].dot_id] == 0
    assert all(0 <= v <= 2 for v in fan.values())


# ---- evaluation helpers ------------------------------------------------------------------
def test_clean_generation_takes_first_line():
    from dots.evaluate import clean_generation
    assert clean_generation(" Paris.\nQuestion: next") == "Paris."
    assert clean_generation("\n\n") == ""


def test_perplexity_matches_manual_cross_entropy(model):
    from dots.evaluate import perplexity
    x = toks(2, 9, seed=5)
    with torch.no_grad():
        logits = model(x).logits
    ce = torch.nn.functional.cross_entropy(logits[:, :-1].reshape(-1, 300), x[:, 1:].reshape(-1))
    assert perplexity(model, [x]) == pytest.approx(float(torch.exp(ce)), rel=1e-4)


def test_prompt_prefix_is_prepended_for_eval_only():
    from dots.qa import FEWSHOT, build_prompt
    ex = QAExample("q?", ["a"])
    assert build_prompt(ex, FEWSHOT).startswith(FEWSHOT) and build_prompt(ex, FEWSHOT).endswith(ex.prompt())
    assert build_prompt(ex) == ex.prompt()


def test_exact_removal_holds_with_per_dot_adam(model):
    ledger = Ledger(model.dots_memory)
    w = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=3, lr=0.05, batch_size=2, optimizer="adam"))
    a = w.learn(facts(0), "A", ledger)
    w.learn(facts(10), "B", ledger)
    ledger.remove(a.dot_id, writer=w, exact=True)
    after = model.dots_memory.values.clone()
    ledger.reset()
    w.learn(facts(10), "B", ledger)
    assert torch.equal(after, model.dots_memory.values)


def test_calibrate_ignores_padding_positions():
    m1, m2 = tiny_model(), tiny_model()
    for m in (m1, m2):
        attach_memory(m, LAYER, HDCMemory(64, n_sub=16, key_dim=16, topk=4))
    a, b = toks(1, 10, seed=7), toks(1, 6, seed=8)
    calibrate(m1, [a, b])
    padded = torch.cat([b, torch.zeros(1, 4, dtype=torch.long)], 1)
    valid = torch.cat([torch.ones(1, 6, dtype=torch.bool), torch.zeros(1, 4, dtype=torch.bool)], 1)
    calibrate(m2, [a, (padded, valid)])
    assert torch.equal(m1.dots_memory.bg_df, m2.dots_memory.bg_df)
    assert torch.allclose(m1.dots_memory.mu, m2.dots_memory.mu, atol=1e-6)


def test_calibration_records_background_read_mass(model):
    mem = model.dots_memory
    assert mem.bg_mass.sum() > 0
    assert torch.all((mem.bg_mass > 0) == (mem.bg_df > 0))


def test_protect_quantile_never_writes_heavily_used_slots(model):
    mem = model.dots_memory
    ledger = Ledger(mem)
    w = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=50, steps=1, lr=0.1, batch_size=2,
                                                       optimizer="adam", protect_quantile=0.5))
    d = w.learn(facts(0), "A", ledger)
    cut = torch.quantile(mem.bg_mass, 0.5)
    assert len(d.write_slots) > 0 and torch.all(mem.bg_mass[d.write_slots] <= cut)


def test_anchor_penalty_shrinks_updates_and_keeps_exactness(model):
    mem = model.dots_memory
    ledger = Ledger(mem)
    free = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=3, lr=0.05, batch_size=2, optimizer="adam"))
    tied = DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=12, steps=3, lr=0.05, batch_size=2, optimizer="adam",
                                                          anchor_weight=1000.0))
    n_free = free.learn(facts(0), "A", ledger).delta.norm()
    ledger.reset()
    a = tied.learn(facts(0), "A", ledger)
    assert a.delta.norm() < n_free
    tied.learn(facts(10), "B", ledger)
    ledger.remove(a.dot_id, writer=tied, exact=True)
    after = mem.values.clone()
    ledger.reset()
    tied.learn(facts(10), "B", ledger)
    assert torch.equal(after, mem.values)


def test_writes_run_with_deterministic_algorithms(model):
    seen = []
    h = model.dots_memory.register_forward_pre_hook(lambda m, a: seen.append(torch.are_deterministic_algorithms_enabled()))
    try:
        DotWriter(model, FakeTok(), LAYER, WriterConfig(write_slots=8, steps=2, lr=0.1, batch_size=2)).learn(
            facts(0), "A", Ledger(model.dots_memory))
    finally:
        h.remove()
    assert any(seen) and not torch.are_deterministic_algorithms_enabled()


def test_prefix_augmentation_builds_context_variants():
    from dots.qa import augment
    ex = QAExample("q?", ["a"])
    out = augment([ex], ["P1\n", "P2\n"])
    assert [e.prompt() for e in out] == [ex.prompt(), "P1\n" + ex.prompt(), "P2\n" + ex.prompt()]
    assert all(e.target() == ex.target() for e in out)


In [ ]:
# Embedded data: 1,000 facts to learn, 300 known facts, 300 NQ-open questions (selected with the base model)
import base64, gzip
DATA_B64 = '...'
open('data/phase1_data.json', 'wb').write(gzip.decompress(base64.b64decode(DATA_B64)))
!python make_chunks.py 2>&1 | grep --line-buffered -v -i -E "warn|loading|it/s|^\s*$"
import torch; c = torch.load('data/wikitext_chunks.pt')
print('chunk checksums match local run:', int(c['background'].sum()) == 438013084 and int(c['heldout'].sum()) == 112696201)

In [ ]:
# 1) Unit tests (CPU, ~1 min)
!python -m pytest -q tests 2>&1 | tail -3

In [ ]:
# 2) Determinism check on this GPU: learn the same Dot twice, deltas must be bit-identical
import sys; sys.path.insert(0, '/content/phase1')
from dots.common import attach_calibrated, load_base, load_data
from dots.ledger import Ledger
from dots.writer import DotWriter, WriterConfig
m, tok = load_base(); data = load_data()
attach_calibrated(m, data, layer=15, tok=tok)
led = Ledger(m.dots_memory)
w = DotWriter(m, tok, 15, WriterConfig(write_slots=3000, steps=5, lr=0.03, optimizer='adam', protect_quantile=0.9))
d1 = w.learn(data['teach'][:16], 'x', led); led.reset(); d2 = w.learn(data['teach'][:16], 'x', led); led.reset()
print('GPU write is bit-reproducible:', torch.equal(d1.write_slots, d2.write_slots) and torch.equal(d1.delta, d2.delta))
del m, w, led, d1, d2; import gc; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 3) Smoke test (tiny sizes, ~2 min)
!python run_dots.py --lr 0.03 --protect 0.9 --slots 3000 --augment --smoke 2>&1 | grep --line-buffered -v -i -E "warn|loading|it/s|^\s*$" | tail -3

In [ ]:
# 4) DOTS v2: each fact also trained under 2 demonstration contexts; tested under an UNSEEN 3rd (~20 min)
!python run_dots.py --lr 0.03 --steps 240 --slots 6000 --opt adam --protect 0.9 --augment --tag aug 2>&1 | grep --line-buffered -v -i -E "warn|loading|it/s|^\s*$"

In [ ]:
# 5) Fair baselines: gentler learning rates, plain data (~25 min)
for m, lr in (('lora', '1e-4'), ('lora', '3e-5'), ('fullft', '1e-5'), ('fullft', '5e-6')):
    print('====', m, lr)
    !python run_baselines.py --method {m} --lr {lr} 2>&1 | grep --line-buffered -v -i -E "warn|loading|it/s|^\s*$" | grep -E 'after all|right after'

In [ ]:
# 6) Baselines with the same context augmentation as DOTS v2 (~20 min)
for m, lr in (('lora', '1e-4'), ('fullft', '1e-5')):
    print('====', m, lr, 'aug')
    !python run_baselines.py --method {m} --lr {lr} --steps 120 --augment 2>&1 | grep --line-buffered -v -i -E "warn|loading|it/s|^\s*$" | grep -E 'after all|right after'

In [ ]:
# 7) Summary + download
import json, os, glob, shutil, pandas as pd
rows = []
for f in sorted(glob.glob('runs/*.json')):
    if 'smoke' in f: continue
    r = json.load(open(f)); a = r.get('after_all')
    if not a: continue
    rows.append({'run': os.path.basename(f)[:-5], 'taught EM': a['teach_em'], 'taught EM unseen context': a['teach_fewshot_dots1_2']['em'],
                 'known kept': a['known']['em'], 'NQ F1': round(a['nq']['f1'], 4), 'ppl': a['ppl'],
                 'exact removal': r.get('exact_equals_never_learned', '')})
display(pd.DataFrame(rows))
os.makedirs('results', exist_ok=True)
for f in glob.glob('runs/*.json'): shutil.copy(f, 'results/')
shutil.make_archive('phase1_v2_results', 'zip', 'results')
from google.colab import files; files.download('phase1_v2_results.zip')